# LiteRT-LM non-f16 fallback shaders on a Colab T4

**First: Runtime → Change runtime type → T4 GPU.**

On Apple Silicon the WebGPU adapter always offers `shader-f16`, so the engine never takes its non-f16 fallback path there (see `ledger/capture-08-parked.json` in the repo for the probe evidence). A T4's Chrome/WebGPU stack withholds `shader-f16`, which forces the f32 fallback — this notebook captures those fallback shaders exactly the way `node capture.mjs` captures the f16 ones on the Mac.

What it does: installs Node + the NVIDIA Vulkan userspace (zero-tvm's recipe, after Chrome's colab-headless guide), downloads the pinned model bundle (2 GB, SHA-checked), serves the stock `www/capture.html` plus the model with COOP/COEP, runs it through headless Chrome, asserts `shader-f16` is absent, and saves `nonf16/nonf16-t4-capture.json` plus `nonf16/shaders/*.wgsl` with per-shader SHA-256.

Nothing is uploaded anywhere. Copy-back instructions are in the last cell. Method, harness and records: https://github.com/abgnydn/litert-capture. Independent work, not affiliated with Google.

In [ ]:
# 1) Node 22 + the NVIDIA Vulkan userspace matching the driver
import subprocess
drv = subprocess.check_output('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True).decode().split('.')[0].strip()
print('NVIDIA driver major:', drv, '-> installing libnvidia-gl-' + drv)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq update > /dev/null
# libasound2 is libasound2t64 on Colab's Ubuntu 24.04; one missing package name aborts the whole install
!apt-get -qq install -y nodejs vulkan-tools libnvidia-gl-{drv} libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2t64 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!rm -f /usr/share/vulkan/icd.d/lvp_icd*.json
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Unpack the capture page + driver, install puppeteer, fetch the pinned model (2 GB, SHA-checked)
import base64, os, hashlib
files = {"capture.html":"PCFkb2N0eXBlIGh0bWw+CjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij4KPHRpdGxlPkxpdGVSVC1MTSB3ZWIgc2hhZGVyIGNhcHR1cmU8L3RpdGxlPgo8cHJlIGlkPSJsb2ciIHN0eWxlPSJ3aGl0ZS1zcGFjZTogcHJlLXdyYXA7IGZvbnQ6IDEycHggbW9ub3NwYWNlIj48L3ByZT4KCjwhLS0gQ2xhc3NpYyBzY3JpcHQ6IHJ1bnMgYmVmb3JlIHRoZSBtb2R1bGUgYmVsb3csIHNvIGV2ZXJ5IFdlYkdQVSBjYWxsIHRoZQogICAgIExpdGVSVC1MTSB3YXNtIG1ha2VzIGdvZXMgdGhyb3VnaCB0aGVzZSB3cmFwcGVycy4gUHJvdG90eXBlLWxldmVsLCBzbyBpdAogICAgIGNvdmVycyBhbnkgR1BVRGV2aWNlLCBob3dldmVyIGl0IGlzIGNyZWF0ZWQuIC0tPgo8c2NyaXB0PgooKCkgPT4gewogIGNvbnN0IGNhcCA9ICh3aW5kb3cuX19jYXB0dXJlID0gewogICAgc2hhZGVyczogW10sICAgICAgLy8geyBpZCwgbGFiZWwsIGNvZGUgfQogICAgcGlwZWxpbmVzOiBbXSwgICAgLy8geyBpZCwgc2hhZGVySWQsIGVudHJ5UG9pbnQsIGxhYmVsLCBhc3luYyB9CiAgICBkaXNwYXRjaEJ5UGlwZWxpbmU6IHt9LCAvLyBwaXBlbGluZUlkIC0+IGNvdW50CiAgICBjb3VudGVyczogeyBkaXNwYXRjaDogMCwgZGlzcGF0Y2hJbmRpcmVjdDogMCwgcGFzc2VzOiAwLCBzdWJtaXRzOiAwLCB3cml0ZUJ1ZmZlcjogMCwgbWFwQXN5bmM6IDAgfSwKICAgIG1hcmtzOiBbXSwgICAgICAgIC8vIHsgbmFtZSwgdCwgY291bnRlcnMsIGRpc3BhdGNoQnlQaXBlbGluZSB9CiAgfSkKICB3aW5kb3cuX19tYXJrID0gKG5hbWUpID0+CiAgICBjYXAubWFya3MucHVzaCh7CiAgICAgIG5hbWUsCiAgICAgIHQ6IHBlcmZvcm1hbmNlLm5vdygpLAogICAgICBjb3VudGVyczogeyAuLi5jYXAuY291bnRlcnMgfSwKICAgICAgZGlzcGF0Y2hCeVBpcGVsaW5lOiB7IC4uLmNhcC5kaXNwYXRjaEJ5UGlwZWxpbmUgfSwKICAgIH0pCgogIGNvbnN0IHNoYWRlcklkcyA9IG5ldyBXZWFrTWFwKCkKICBjb25zdCBwaXBlbGluZUlkcyA9IG5ldyBXZWFrTWFwKCkKICBjb25zdCBwYXNzUGlwZWxpbmUgPSBuZXcgV2Vha01hcCgpCgogIC8vIFNIQS0yNTYgb2YgdGhlIFdHU0wgdGV4dCBpcyB0aGUgc2hhZGVyJ3MgaWRlbnRpdHkgYWNyb3NzIHJ1bnM7IGJ5dGUgbGVuZ3RoCiAgLy8gaXMgbm90IChkaXN0aW5jdCBzaGFkZXJzIGhlcmUgZG8gc2hhcmUgbGVuZ3RocykuIGRpZ2VzdCgpIGlzIGFzeW5jIGFuZAogIC8vIGNyZWF0ZVNoYWRlck1vZHVsZSBpcyBub3QsIHNvIHRoZSBoYXNoZXMgYXJlIGZpbGxlZCBpbiBhZnRlcndhcmRzIGFuZCB0aGUKICAvLyBwYWdlIHdhaXRzIGZvciB0aGVtIGJlZm9yZSBkZWNsYXJpbmcgaXRzZWxmIGRvbmUuIElmIGNyeXB0by5zdWJ0bGUgaXMKICAvLyBtaXNzaW5nIHRoZSByZWNvcmQgc2F5cyBzbyByYXRoZXIgdGhhbiBjYXJyeWluZyBhIG1hZGUtdXAgaGFzaC4KICBjb25zdCBoYXNoSm9icyA9IFtdCiAgY29uc3QgaGFzaEludG8gPSAocmVjLCB0ZXh0KSA9PiB7CiAgICBpZiAoIWdsb2JhbFRoaXMuY3J5cHRvPy5zdWJ0bGUpIHsgY2FwLnNoYWRlclNoYTI1NlVuYXZhaWxhYmxlID0gdHJ1ZTsgcmV0dXJuIH0KICAgIGhhc2hKb2JzLnB1c2goY3J5cHRvLnN1YnRsZS5kaWdlc3QoJ1NIQS0yNTYnLCBuZXcgVGV4dEVuY29kZXIoKS5lbmNvZGUodGV4dCkpLnRoZW4oKGQpID0+IHsKICAgICAgcmVjLnNoYWRlclNoYTI1NiA9IFsuLi5uZXcgVWludDhBcnJheShkKV0ubWFwKChiKSA9PiBiLnRvU3RyaW5nKDE2KS5wYWRTdGFydCgyLCAnMCcpKS5qb2luKCcnKQogICAgfSkpCiAgfQogIHdpbmRvdy5fX3NldHRsZUhhc2hlcyA9ICgpID0+IFByb21pc2UuYWxsKGhhc2hKb2JzKQoKICBjb25zdCB3cmFwID0gKHByb3RvLCBuYW1lLCBmbikgPT4gewogICAgY29uc3Qgb3JpZyA9IHByb3RvW25hbWVdCiAgICBwcm90b1tuYW1lXSA9IGZ1bmN0aW9uICguLi5hcmdzKSB7IHJldHVybiBmbi5jYWxsKHRoaXMsIG9yaWcsIGFyZ3MpIH0KICB9CgogIHdyYXAoR1BVRGV2aWNlLnByb3RvdHlwZSwgJ2NyZWF0ZVNoYWRlck1vZHVsZScsIGZ1bmN0aW9uIChvcmlnLCBhcmdzKSB7CiAgICBjb25zdCBtb2QgPSBvcmlnLmFwcGx5KHRoaXMsIGFyZ3MpCiAgICBjb25zdCBpZCA9IGNhcC5zaGFkZXJzLmxlbmd0aAogICAgY29uc3QgcmVjID0geyBpZCwgbGFiZWw6IGFyZ3NbMF0/LmxhYmVsID8/ICcnLCBjb2RlOiBhcmdzWzBdPy5jb2RlID8/ICcnLCBzaGFkZXJTaGEyNTY6IG51bGwgfQogICAgY2FwLnNoYWRlcnMucHVzaChyZWMpCiAgICBoYXNoSW50byhyZWMsIHJlYy5jb2RlKQogICAgc2hhZGVySWRzLnNldChtb2QsIGlkKQogICAgcmV0dXJuIG1vZAogIH0pCgogIGNvbnN0IHJlY29yZFBpcGVsaW5lID0gKHBpcGVsaW5lLCBkZXNjLCBpc0FzeW5jKSA9PiB7CiAgICBjb25zdCBpZCA9IGNhcC5waXBlbGluZXMubGVuZ3RoCiAgICBjYXAucGlwZWxpbmVzLnB1c2goewogICAgICBpZCwKICAgICAgc2hhZGVySWQ6IHNoYWRlcklkcy5nZXQoZGVzYz8uY29tcHV0ZT8ubW9kdWxlKSA/PyAtMSwKICAgICAgZW50cnlQb2ludDogZGVzYz8uY29tcHV0ZT8uZW50cnlQb2ludCA/PyAnJywKICAgICAgbGFiZWw6IGRlc2M/LmxhYmVsID8/ICcnLAogICAgICBjb25zdGFudHM6IGRlc2M/LmNvbXB1dGU/LmNvbnN0YW50cyA/PyBudWxsLAogICAgICBhc3luYzogaXNBc3luYywKICAgIH0pCiAgICBwaXBlbGluZUlkcy5zZXQocGlwZWxpbmUsIGlkKQogIH0KICB3cmFwKEdQVURldmljZS5wcm90b3R5cGUsICdjcmVhdGVDb21wdXRlUGlwZWxpbmUnLCBmdW5jdGlvbiAob3JpZywgYXJncykgewogICAgY29uc3QgcCA9IG9yaWcuYXBwbHkodGhpcywgYXJncykKICAgIHJlY29yZFBpcGVsaW5lKHAsIGFyZ3NbMF0sIGZhbHNlKQogICAgcmV0dXJuIHAKICB9KQogIHdyYXAoR1BVRGV2aWNlLnByb3RvdHlwZSwgJ2NyZWF0ZUNvbXB1dGVQaXBlbGluZUFzeW5jJywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIHJldHVybiBvcmlnLmFwcGx5KHRoaXMsIGFyZ3MpLnRoZW4oKHApID0+IHsgcmVjb3JkUGlwZWxpbmUocCwgYXJnc1swXSwgdHJ1ZSk7IHJldHVybiBwIH0pCiAgfSkKCiAgd3JhcChHUFVDb21tYW5kRW5jb2Rlci5wcm90b3R5cGUsICdiZWdpbkNvbXB1dGVQYXNzJywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIGNhcC5jb3VudGVycy5wYXNzZXMrKwogICAgcmV0dXJuIG9yaWcuYXBwbHkodGhpcywgYXJncykKICB9KQogIHdyYXAoR1BVQ29tcHV0ZVBhc3NFbmNvZGVyLnByb3RvdHlwZSwgJ3NldFBpcGVsaW5lJywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIHBhc3NQaXBlbGluZS5zZXQodGhpcywgcGlwZWxpbmVJZHMuZ2V0KGFyZ3NbMF0pID8/IC0xKQogICAgcmV0dXJuIG9yaWcuYXBwbHkodGhpcywgYXJncykKICB9KQogIGNvbnN0IGNvdW50RGlzcGF0Y2ggPSAocGFzcywga2V5KSA9PiB7CiAgICBjYXAuY291bnRlcnNba2V5XSsrCiAgICBjb25zdCBwaWQgPSBwYXNzUGlwZWxpbmUuZ2V0KHBhc3MpID8/IC0xCiAgICBjYXAuZGlzcGF0Y2hCeVBpcGVsaW5lW3BpZF0gPSAoY2FwLmRpc3BhdGNoQnlQaXBlbGluZVtwaWRdID8/IDApICsgMQogIH0KICB3cmFwKEdQVUNvbXB1dGVQYXNzRW5jb2Rlci5wcm90b3R5cGUsICdkaXNwYXRjaFdvcmtncm91cHMnLCBmdW5jdGlvbiAob3JpZywgYXJncykgewogICAgY291bnREaXNwYXRjaCh0aGlzLCAnZGlzcGF0Y2gnKQogICAgcmV0dXJuIG9yaWcuYXBwbHkodGhpcywgYXJncykKICB9KQogIHdyYXAoR1BVQ29tcHV0ZVBhc3NFbmNvZGVyLnByb3RvdHlwZSwgJ2Rpc3BhdGNoV29ya2dyb3Vwc0luZGlyZWN0JywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIGNvdW50RGlzcGF0Y2godGhpcywgJ2Rpc3BhdGNoSW5kaXJlY3QnKQogICAgcmV0dXJuIG9yaWcuYXBwbHkodGhpcywgYXJncykKICB9KQogIHdyYXAoR1BVUXVldWUucHJvdG90eXBlLCAnc3VibWl0JywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIGNhcC5jb3VudGVycy5zdWJtaXRzKysKICAgIHJldHVybiBvcmlnLmFwcGx5KHRoaXMsIGFyZ3MpCiAgfSkKICB3cmFwKEdQVVF1ZXVlLnByb3RvdHlwZSwgJ3dyaXRlQnVmZmVyJywgZnVuY3Rpb24gKG9yaWcsIGFyZ3MpIHsKICAgIGNhcC5jb3VudGVycy53cml0ZUJ1ZmZlcisrCiAgICByZXR1cm4gb3JpZy5hcHBseSh0aGlzLCBhcmdzKQogIH0pCiAgd3JhcChHUFVCdWZmZXIucHJvdG90eXBlLCAnbWFwQXN5bmMnLCBmdW5jdGlvbiAob3JpZywgYXJncykgewogICAgY2FwLmNvdW50ZXJzLm1hcEFzeW5jKysKICAgIHJldHVybiBvcmlnLmFwcGx5KHRoaXMsIGFyZ3MpCiAgfSkKfSkoKQo8L3NjcmlwdD4KCjxzY3JpcHQgdHlwZT0ibW9kdWxlIj4KICBjb25zdCBsb2cgPSAocykgPT4geyBkb2N1bWVudC5nZXRFbGVtZW50QnlJZCgnbG9nJykuYXBwZW5kKHMgKyAnXG4nKTsgY29uc29sZS5sb2coJ1tjYXB0dXJlXSAnICsgcykgfQogIGNvbnN0IHBhcmFtcyA9IG5ldyBVUkxTZWFyY2hQYXJhbXMobG9jYXRpb24uc2VhcmNoKQogIGNvbnN0IFZFUlNJT04gPSBwYXJhbXMuZ2V0KCd2JykgPz8gJzAuMTcuMScKICBjb25zdCBNQVhfQ0hVTktTID0gTnVtYmVyKHBhcmFtcy5nZXQoJ2NodW5rcycpID8/IDQ4KQogIC8vIFJPQURNQVAgMDc6IGhlYWRsZXNzIGhvb2tzLiA/YXV0b3N0YXJ0PTAgYXJtcyBpbnN0ZWFkIG9mIGF1dG8tcnVubmluZyBzbyBhCiAgLy8gZHJpdmVyIGNhbiBhdHRhY2ggbGlzdGVuZXJzIGZpcnN0LCB0aGVuIGNhbGwgd2luZG93Ll9fY2FwdHVyZVN0YXJ0KCkuCiAgLy8gd2luZG93Ll9fY2FwdHVyZVN0b3AoKSBmb3JjZXMgX19jYXB0dXJlRG9uZSBmb3IgZWFybHkgc3RvcC4gRGVmYXVsdAogIC8vIGF1dG9zdGFydCBrZWVwcyBleGlzdGluZyBgbm9kZSBjYXB0dXJlLm1qc2AgYmVoYXZpb3IgdW5jaGFuZ2VkLgogIGNvbnN0IEFVVE9TVEFSVCA9IHBhcmFtcy5nZXQoJ2F1dG9zdGFydCcpICE9PSAnMCcKICBsZXQgc3RhcnRlZCA9IGZhbHNlCiAgYXN5bmMgZnVuY3Rpb24gcnVuQ2FwdHVyZSgpIHsKICAgIGlmIChzdGFydGVkKSByZXR1cm4KICAgIHN0YXJ0ZWQgPSB0cnVlCiAgICB0cnkgewogICAgICBjb25zdCBhZGFwdGVyID0gYXdhaXQgbmF2aWdhdG9yLmdwdT8ucmVxdWVzdEFkYXB0ZXIoKQogICAgICB3aW5kb3cuX19jYXB0dXJlLmFkYXB0ZXIgPSBhZGFwdGVyCiAgICAgICAgPyB7IHZlbmRvcjogYWRhcHRlci5pbmZvPy52ZW5kb3IsIGFyY2hpdGVjdHVyZTogYWRhcHRlci5pbmZvPy5hcmNoaXRlY3R1cmUsIGRldmljZTogYWRhcHRlci5pbmZvPy5kZXZpY2UsIGRlc2NyaXB0aW9uOiBhZGFwdGVyLmluZm8/LmRlc2NyaXB0aW9uLCBmZWF0dXJlczogWy4uLmFkYXB0ZXIuZmVhdHVyZXNdIH0KICAgICAgICA6IG51bGwKICAgICAgbG9nKCdhZGFwdGVyOiAnICsgSlNPTi5zdHJpbmdpZnkod2luZG93Ll9fY2FwdHVyZS5hZGFwdGVyKSkKICAgICAgbG9nKCdjcm9zc09yaWdpbklzb2xhdGVkOiAnICsgc2VsZi5jcm9zc09yaWdpbklzb2xhdGVkKQoKICAgICAgY29uc3QgeyBFbmdpbmUgfSA9IGF3YWl0IGltcG9ydChgaHR0cHM6Ly9jZG4uanNkZWxpdnIubmV0L25wbS9AbGl0ZXJ0LWxtL2NvcmVAJHtWRVJTSU9OfS8rZXNtYCkKICAgICAgd2luZG93Ll9fY2FwdHVyZS52ZXJzaW9uID0gVkVSU0lPTgogICAgICBfX21hcmsoJ2ltcG9ydC1kb25lJykKCiAgICAgIGNvbnN0IGVuZ2luZSA9IGF3YWl0IEVuZ2luZS5jcmVhdGUoeyBtb2RlbDogJy9nZW1tYS00LUUyQi1pdC13ZWIubGl0ZXJ0bG0nIH0pCiAgICAgIF9fbWFyaygnZW5naW5lLWNyZWF0ZWQnKQogICAgICBsb2coYGVuZ2luZSBjcmVhdGVkOiAke3dpbmRvdy5fX2NhcHR1cmUuc2hhZGVycy5sZW5ndGh9IHNoYWRlcnMsICR7d2luZG93Ll9fY2FwdHVyZS5waXBlbGluZXMubGVuZ3RofSBwaXBlbGluZXMgc28gZmFyYCkKCiAgICAgIGNvbnN0IGNoYXQgPSBhd2FpdCBlbmdpbmUuY3JlYXRlQ29udmVyc2F0aW9uKCkKICAgICAgX19tYXJrKCdjb252ZXJzYXRpb24tY3JlYXRlZCcpCgogICAgICBsZXQgdGV4dCA9ICcnCiAgICAgIGxldCBjaHVua3MgPSAwCiAgICAgIGZvciBhd2FpdCAoY29uc3QgY2h1bmsgb2YgY2hhdC5zZW5kTWVzc2FnZVN0cmVhbWluZygnQ291bnQgZnJvbSBvbmUgdG8gdGhpcnR5IGluIEVuZ2xpc2ggd29yZHMsIHNlcGFyYXRlZCBieSBjb21tYXMuJykpIHsKICAgICAgICBjb25zdCBwaWVjZSA9IGNodW5rPy5jb250ZW50Py5bMF0/LnRleHQgPz8gJycKICAgICAgICB0ZXh0ICs9IHBpZWNlCiAgICAgICAgX19tYXJrKGBjaHVuay0ke2NodW5rc31gKQogICAgICAgIGlmICgrK2NodW5rcyA+PSBNQVhfQ0hVTktTKSBicmVhawogICAgICB9CiAgICAgIF9fbWFyaygnZ2VuZXJhdGlvbi1kb25lJykKICAgICAgd2luZG93Ll9fY2FwdHVyZS5vdXRwdXQgPSB0ZXh0CiAgICAgIGxvZyhgb3V0cHV0ICgke2NodW5rc30gY2h1bmtzKTogJHt0ZXh0fWApCiAgICAgIGxvZyhgdG90YWw6ICR7d2luZG93Ll9fY2FwdHVyZS5zaGFkZXJzLmxlbmd0aH0gc2hhZGVycywgJHt3aW5kb3cuX19jYXB0dXJlLnBpcGVsaW5lcy5sZW5ndGh9IHBpcGVsaW5lcywgJHt3aW5kb3cuX19jYXB0dXJlLmNvdW50ZXJzLmRpc3BhdGNofSBkaXNwYXRjaGVzYCkKICAgIH0gY2F0Y2ggKGUpIHsKICAgICAgd2luZG93Ll9fY2FwdHVyZS5lcnJvciA9IFN0cmluZyhlPy5zdGFjayA/PyBlKQogICAgICBsb2coJ0VSUk9SOiAnICsgd2luZG93Ll9fY2FwdHVyZS5lcnJvcikKICAgIH0KICAgIGF3YWl0IHdpbmRvdy5fX3NldHRsZUhhc2hlcygpCiAgICBpZiAod2luZG93Ll9fY2FwdHVyZS5zaGFkZXJTaGEyNTZVbmF2YWlsYWJsZSkgbG9nKCdXQVJOSU5HOiBjcnlwdG8uc3VidGxlIHVuYXZhaWxhYmxlLCBzaGFkZXIgaGFzaGVzIG5vdCByZWNvcmRlZCBpbiB0aGUgcGFnZScpCiAgICB3aW5kb3cuX19jYXB0dXJlRG9uZSA9IHRydWUKICB9CiAgd2luZG93Ll9fY2FwdHVyZVN0YXJ0ID0gKCkgPT4gcnVuQ2FwdHVyZSgpCiAgd2luZG93Ll9fY2FwdHVyZVN0b3AgPSAoKSA9PiB7IHdpbmRvdy5fX2NhcHR1cmVEb25lID0gdHJ1ZSB9CiAgaWYgKEFVVE9TVEFSVCkgcnVuQ2FwdHVyZSgpCiAgZWxzZSBsb2coJ2NhcHR1cmUgYXJtZWQgKD9hdXRvc3RhcnQ9MCk6IGNhbGwgd2luZG93Ll9fY2FwdHVyZVN0YXJ0KCknKQo8L3NjcmlwdD4K","run.mjs":"Ly8gaGVhZGxlc3MgQ2hyb21lIG9uIGEgQ29sYWIgVDQ6IHNlcnZlcyBjYXB0dXJlLmh0bWwgKyB0aGUgbW9kZWwgYnVuZGxlIHdpdGgKLy8gQ09PUC9DT0VQLCBydW5zIHRoZSBzdG9jayBjYXB0dXJlLCBhc3NlcnRzIHRoZSBhZGFwdGVyIGhhcyBubyBzaGFkZXItZjE2Ci8vIChzbyB0aGUgZW5naW5lIHRvb2sgaXRzIGZhbGxiYWNrIHBhdGgpLCBhbmQgc2F2ZXMgdGhlIGZhbGxiYWNrIHNoYWRlcnMuCmltcG9ydCB7IGNyZWF0ZUhhc2ggfSBmcm9tICdub2RlOmNyeXB0bycKaW1wb3J0IHsgY3JlYXRlUmVhZFN0cmVhbSwgbWtkaXJTeW5jLCBzdGF0U3luYywgd3JpdGVGaWxlU3luYyB9IGZyb20gJ25vZGU6ZnMnCmltcG9ydCB7IGNyZWF0ZVNlcnZlciB9IGZyb20gJ25vZGU6aHR0cCcKaW1wb3J0IHsgZGlybmFtZSwgZXh0bmFtZSwgam9pbiwgbm9ybWFsaXplIH0gZnJvbSAnbm9kZTpwYXRoJwppbXBvcnQgeyBmaWxlVVJMVG9QYXRoIH0gZnJvbSAnbm9kZTp1cmwnCmltcG9ydCBwdXBwZXRlZXIgZnJvbSAncHVwcGV0ZWVyJwpjb25zdCBST09UID0gcHJvY2Vzcy5jd2QoKQpjb25zdCBNSU1FID0geyAnLmh0bWwnOiAndGV4dC9odG1sOyBjaGFyc2V0PXV0Zi04JywgJy5saXRlcnRsbSc6ICdhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0nIH0KY29uc3Qgc2VydmVyID0gY3JlYXRlU2VydmVyKChyZXEsIHJlcykgPT4gewogIGNvbnN0IGYgPSBqb2luKFJPT1QsIG5vcm1hbGl6ZShkZWNvZGVVUklDb21wb25lbnQobmV3IFVSTChyZXEudXJsLCAnaHR0cDovL3gnKS5wYXRobmFtZSkpKQogIGlmICghZi5zdGFydHNXaXRoKFJPT1QpKSByZXR1cm4gcmVzLndyaXRlSGVhZCg0MDMpLmVuZCgpCiAgbGV0IHM7IHRyeSB7IHMgPSBzdGF0U3luYyhmKS5zaXplIH0gY2F0Y2ggeyByZXR1cm4gcmVzLndyaXRlSGVhZCg0MDQpLmVuZCgpIH0KICByZXMud3JpdGVIZWFkKDIwMCwgeyAnQ29udGVudC1UeXBlJzogTUlNRVtleHRuYW1lKGYpXSA/PyAnYXBwbGljYXRpb24vb2N0ZXQtc3RyZWFtJywgJ0NvbnRlbnQtTGVuZ3RoJzogcywgJ0Nyb3NzLU9yaWdpbi1PcGVuZXItUG9saWN5JzogJ3NhbWUtb3JpZ2luJywgJ0Nyb3NzLU9yaWdpbi1FbWJlZGRlci1Qb2xpY3knOiAncmVxdWlyZS1jb3JwJyB9KQogIGNyZWF0ZVJlYWRTdHJlYW0oZikucGlwZShyZXMpCn0pCmF3YWl0IG5ldyBQcm9taXNlKChyKSA9PiBzZXJ2ZXIubGlzdGVuKDg5MTcsICcxMjcuMC4wLjEnLCByKSkKY29uc3QgYnJvd3NlciA9IGF3YWl0IHB1cHBldGVlci5sYXVuY2goeyBoZWFkbGVzczogJ25ldycsIGFyZ3M6IFsKICAnLS1uby1zYW5kYm94JywgJy0tdXNlLWFuZ2xlPXZ1bGthbicsICctLWVuYWJsZS1mZWF0dXJlcz1WdWxrYW4nLCAnLS1kaXNhYmxlLXZ1bGthbi1zdXJmYWNlJywKICAnLS1lbmFibGUtdW5zYWZlLXdlYmdwdScsICctLWVuYWJsZS1kYXduLWZlYXR1cmVzPWFsbG93X3Vuc2FmZV9hcGlzLGRpc2FibGVfYWRhcHRlcl9ibG9ja2xpc3QnLAogICctLWRpc2FibGUtZGF3bi1mZWF0dXJlcz10aW1lc3RhbXBfcXVhbnRpemF0aW9uJywgJy0taWdub3JlLWdwdS1ibG9ja2xpc3QnIF0sIHByb3RvY29sVGltZW91dDogMTIwMDAwMCB9KQpjb25zdCBwYWdlID0gYXdhaXQgYnJvd3Nlci5uZXdQYWdlKCkKcGFnZS5vbignY29uc29sZScsIChtKSA9PiB7IGNvbnN0IHQgPSBtLnRleHQoKTsgaWYgKHQuc3RhcnRzV2l0aCgnW2NhcHR1cmVdJykpIGNvbnNvbGUubG9nKHQpIH0pCmF3YWl0IHBhZ2UuZ290bygnaHR0cDovLzEyNy4wLjAuMTo4OTE3L2NhcHR1cmUuaHRtbCcsIHsgd2FpdFVudGlsOiAnbG9hZCcgfSkKYXdhaXQgcGFnZS53YWl0Rm9yRnVuY3Rpb24oJ3dpbmRvdy5fX2NhcHR1cmVEb25lID09PSB0cnVlJywgeyB0aW1lb3V0OiAxMjAwMDAwLCBwb2xsaW5nOiAxMDAwIH0pCmNvbnN0IGNhcCA9IGF3YWl0IHBhZ2UuZXZhbHVhdGUoKCkgPT4geyBjb25zdCB7IGFkYXB0ZXIsIC4uLnJlc3QgfSA9IHdpbmRvdy5fX2NhcHR1cmU7IHJldHVybiB7IGFkYXB0ZXIsIC4uLnJlc3QgfSB9KQpjb25zdCBmZWF0cyA9IGNhcC5hZGFwdGVyPy5mZWF0dXJlcyA/PyBbXQpjb25zb2xlLmxvZygnQURBUFRFUicsIEpTT04uc3RyaW5naWZ5KGNhcC5hZGFwdGVyKSkKY29uc29sZS5sb2coJ3NoYWRlci1mMTYgcHJlc2VudDonLCBmZWF0cy5pbmNsdWRlcygnc2hhZGVyLWYxNicpLCAnKHdhbnQgZmFsc2U6IGZhbGxiYWNrIHBhdGgpJykKbWtkaXJTeW5jKCdub25mMTYvc2hhZGVycycsIHsgcmVjdXJzaXZlOiB0cnVlIH0pCmZvciAoY29uc3QgcyBvZiBjYXAuc2hhZGVycykgewogIGNvbnN0IHNhZmUgPSAocy5sYWJlbCB8fCAndW5sYWJlbGVkJykucmVwbGFjZSgvW15BLVphLXowLTlfLi1dKy9nLCAnXycpLnNsaWNlKDAsIDgwKQogIHdyaXRlRmlsZVN5bmMoam9pbignbm9uZjE2L3NoYWRlcnMnLCBTdHJpbmcocy5pZCkucGFkU3RhcnQoNCwgJzAnKSArICdfJyArIHNhZmUgKyAnLndnc2wnKSwgcy5jb2RlKQp9CmNvbnN0IHsgc2hhZGVycywgLi4ubWFuaWZlc3QgfSA9IGNhcAptYW5pZmVzdC5zaGFkZXJzID0gc2hhZGVycy5tYXAoKHsgaWQsIGxhYmVsLCBjb2RlIH0pID0+ICh7IGlkLCBsYWJlbCwgYnl0ZXM6IGNvZGUubGVuZ3RoLCBzaGEyNTY6IGNyZWF0ZUhhc2goJ3NoYTI1NicpLnVwZGF0ZShjb2RlKS5kaWdlc3QoJ2hleCcpIH0pKQptYW5pZmVzdC5kYXRlID0gbmV3IERhdGUoKS50b0lTT1N0cmluZygpCndyaXRlRmlsZVN5bmMoJ25vbmYxNi9ub25mMTYtdDQtY2FwdHVyZS5qc29uJywgSlNPTi5zdHJpbmdpZnkobWFuaWZlc3QsIG51bGwsIDIpKQpjb25zb2xlLmxvZygnc2hhZGVyczonLCBzaGFkZXJzLmxlbmd0aCwgJ3BpcGVsaW5lczonLCBjYXAucGlwZWxpbmVzLmxlbmd0aCwgJ2Rpc3BhdGNoZXM6JywgY2FwLmNvdW50ZXJzLmRpc3BhdGNoKQppZiAoY2FwLmVycm9yKSBjb25zb2xlLmxvZygnUEFHRSBFUlJPUicsIGNhcC5lcnJvcikKY29uc29sZS5sb2coJ3NhdmVkIG5vbmYxNi9ub25mMTYtdDQtY2FwdHVyZS5qc29uICsgbm9uZjE2L3NoYWRlcnMvOyBjb3B5IHRoZW0gYmFjayAoc2VlIGluc3RydWN0aW9ucyBjZWxsKScpCmF3YWl0IGJyb3dzZXIuY2xvc2UoKTsgc2VydmVyLmNsb3NlKCkK"}
os.makedirs('nonf16/shaders', exist_ok=True)
for name, data in files.items():
    p = os.path.join('nonf16', name)
    os.makedirs(os.path.dirname(p) or '.', exist_ok=True)
    with open(p, 'wb') as f:
        f.write(base64.b64decode(data))
!cd nonf16 && npm init -y > /dev/null && npm i --silent puppeteer@24.40.0 > /dev/null && echo installed
!curl -L -C - -o nonf16/gemma-4-E2B-it-web.litertlm https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/b3ca0d2f076785a8f4b2219ddbd2bdb99954eae1/gemma-4-E2B-it-web.litertlm
import hashlib
h = hashlib.sha256()
with open('nonf16/gemma-4-E2B-it-web.litertlm', 'rb') as f:
    [h.update(c) for c in iter(lambda: f.read(1 << 26), b'')]
print('model sha256:', h.hexdigest())
assert h.hexdigest() == '3a08e8d94e23b814ae5414469c370c503813949acb8ceaa17e4ebf8a35af35b5', 'model bundle mismatch, stop here'

In [ ]:
# 3) Run the capture (up to ~20 min: 2 GB model load + 48 decode chunks on T4).
# The driver asserts shader-f16 is absent and prints the fallback counts.
!cd nonf16 && node run.mjs

## Copy-back instructions

1. Download `nonf16/nonf16-t4-capture.json` and the `nonf16/shaders/*.wgsl` files from this runtime.
2. In the litert-capture repo, on a branch from `main`: save the JSON as `out/nonf16.json`, add this run's `manifest_sha256` join (see `shader-sha.mjs`), and diff the fallback shader set against `out/shaders/*.wgsl` (expect: f32 transcriptions where the Mac set uses f16, plus any extra fallback-only pipelines).
3. Run the standard gates (`pnpm exec biome check .`, `node --check`, `node analyze.mjs 400`, `gitleaks`), commit, and merge — that unparks CAPTURE-ROADMAP item 8.
4. If the driver prints `shader-f16 present: true`, the T4 took the f16 path: stop, report the Chrome version and adapter string from the ADAPTER line, do not commit the capture as a fallback record.